In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,141,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,141,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,141,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,141,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,zero,0,129,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,zero,0,129,0.0
539997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,zero,0,129,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,0,129,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  407421.503625
                                  2                  239294.102982
                                  3                  309160.646929
                                  4                  274713.546717
                                  5                  177056.632350
intervention  maternal_disorders  1                  407421.503625
                                  2                  239294.102982
                                  3                  309160.646929
                                  4                  274713.546717
                                  5                  177056.632350
zero          maternal_disorders  1                  420399.893972
                                  2                  246552.950087
                                  3                  315300.234530
                                  4                  280031.108176
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,141,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,141,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,141,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,141,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,0,129,0.0
1889996,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,0,129,0.0
1889997,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,0,129,0.0
1889998,ylds,cause,all_causes,all_causes,95_plus,severe,5,zero,0,129,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   97151.457358
                                  2                   76162.650139
                                  3                   61237.754278
                                  4                   52554.416948
                                  5                   40762.775467
              maternal_disorders  1                   73573.647978
                                  2                   38880.315135
                                  3                   45769.281856
                                  4                   45403.059287
                                  5                   31759.306548
intervention  anemia              1                   97151.457358
                                  2                   76162.650139
                                  3                   61237.754278
                                  4                   52554.416948
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   97151.457358
                                  2                   76162.650139
                                  3                   61237.754278
                                  4                   52554.416948
                                  5                   40762.775467
              maternal_disorders  1                  480995.151603
                                  2                  278174.418117
                                  3                  354929.928786
                                  4                  320116.606005
                                  5                  208815.938898
intervention  anemia              1                   97151.457358
                                  2                   76162.650139
                                  3                   61237.754278
                                  4                   52554.416948
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,37250.227367
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,24689.127790
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,26205.102718
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,27285.881499
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,23819.839841
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,3361.700749
23996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,1888.877462
23997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,1462.079698
23998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,2517.382098


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.788349e+07
                      2                  1.489822e+07
                      3                  1.320481e+07
                      4                  1.234068e+07
                      5                  1.182422e+07
intervention  lbwsg   1                  1.788349e+07
                      2                  1.489822e+07
                      3                  1.320481e+07
                      4                  1.234068e+07
                      5                  1.182422e+07
zero          lbwsg   1                  1.793528e+07
                      2                  1.493809e+07
                      3                  1.323493e+07
                      4                  1.236235e+07
                      5                  1.183418e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1186.490411,zero
1,Female,0.0,0.019178,2,901.389895,zero
2,Female,0.0,0.019178,3,798.971870,zero
3,Female,0.0,0.019178,4,664.417092,zero
4,Female,0.0,0.019178,5,465.048259,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,80.978228,intervention
746,Male,95.0,125.000000,2,75.451506,intervention
747,Male,95.0,125.000000,3,77.357440,intervention
748,Male,95.0,125.000000,4,72.631383,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.810846e+06
                      2                  1.772233e+06
                      3                  1.816683e+06
                      4                  1.734604e+06
                      5                  1.580374e+06
intervention  anemia  1                  1.810846e+06
                      2                  1.772233e+06
                      3                  1.816683e+06
                      4                  1.734604e+06
                      5                  1.580374e+06
zero          anemia  1                  1.935722e+06
                      2                  1.894235e+06
                      3                  1.926864e+06
                      4                  1.828468e+06
                      5                  1.629746e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-529423.3483828027

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  622350.110058
                      2                  507914.038386
                      3                  468649.860085
                      4                  402170.554909
                      5                  314830.860156
intervention  anemia  1                  622350.110058
                      2                  507914.038386
                      3                  468649.860085
                      4                  402170.554909
                      5                  314830.860156
zero          anemia  1                  657232.983445
                      2                  537238.145707
                      3                  492948.098560
                      4                  420532.322873
                      5                  323175.185458
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-115211.31245009555

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.360566e+06
                      2                  3.959792e+06
                      3                  3.915742e+06
                      4                  3.614767e+06
                      5                  3.222801e+06
intervention  anemia  1                  4.360566e+06
                      2                  3.959792e+06
                      3                  3.915742e+06
                      4                  3.614767e+06
                      5                  3.222801e+06
zero          anemia  1                  4.664061e+06
                      2                  4.232154e+06
                      3                  4.153325e+06
                      4                  3.809427e+06
                      5                  3.321418e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  363252.788867
                      2                  320895.928809
                      3                  288089.125399
                      4                  271653.847670
                      5                  231356.178732
baseline      ntd     1                  337620.098439
                      2                  301581.746928
                      3                  273425.096722
                      4                  259634.684758
                      5                  226772.691855
intervention  ntd     1                  191469.498251
                      2                  184057.448337
                      3                  178650.695942
                      4                  178416.939677
                      5                  189908.375758
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.457717e+06
                                  2                  4.035955e+06
                                  3                  3.976980e+06
                                  4                  3.667322e+06
                                  5                  3.263564e+06
              lbwsg               1                  1.788349e+07
                                  2                  1.489822e+07
                                  3                  1.320481e+07
                                  4                  1.234068e+07
                                  5                  1.182422e+07
              maternal_disorders  1                  4.809952e+05
                                  2                  2.781744e+05
                                  3                  3.549299e+05
                                  4                  3.201166e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)